In [0]:
%run ./00_config


In [0]:
from pyspark.sql import functions as F

s = spark.table(f"{SILVER}.health_indicators")
key = F.xxhash64

dim_geography = (s.groupBy("geo").agg(F.first("dguid").alias("dguid"))
                 .select(key("geo").alias("geo_key"), F.col("geo").alias("geography"), "dguid"))
dim_indicator = (s.select("indicator").distinct()
                 .select(key("indicator").alias("indicator_key"), "indicator"))
dim_demographic = (s.select("age_group", "sex").distinct()
                   .select(key("age_group", "sex").alias("demo_key"), "age_group", "sex"))
dim_measure = (s.select("characteristic", "uom").distinct()
               .select(key("characteristic", "uom").alias("measure_key"), "characteristic", "uom"))

latest = s.agg(F.max("year")).first()[0]
dim_year = (s.select("year").distinct()
            .withColumn("is_latest_year", F.col("year") == F.lit(latest)))

fact = s.select(
    key("geo").alias("geo_key"),
    key("indicator").alias("indicator_key"),
    key("age_group", "sex").alias("demo_key"),
    key("characteristic", "uom").alias("measure_key"),
    "year", "value", "status", "symbol")

In [0]:
tables = {
    "dim_geography": dim_geography, "dim_indicator": dim_indicator,
    "dim_demographic": dim_demographic, "dim_measure": dim_measure,
    "dim_year": dim_year, "fact_health_indicator": fact,
}
for name, df in tables.items():
    (df.write.mode("overwrite").option("overwriteSchema", "true")
       .format("delta").saveAsTable(f"{GOLD}.{name}"))
    print(f"{name}: {spark.table(f'{GOLD}.{name}').count():,} rows")

In [0]:
f = spark.table(f"{GOLD}.fact_health_indicator")
for name, k in [("dim_geography", "geo_key"), ("dim_indicator", "indicator_key"),
                ("dim_demographic", "demo_key"), ("dim_measure", "measure_key"),
                ("dim_year", "year")]:
    d = spark.table(f"{GOLD}.{name}")
    assert d.count() == d.select(k).distinct().count(), f"Duplicate keys in {name}"
    orphans = f.join(d, k, "left_anti").count()
    assert orphans == 0, f"{orphans} fact rows have no match in {name}"

spark.sql(f"OPTIMIZE {GOLD}.fact_health_indicator")
print("Gold checks passed")

In [0]:
%sql

SELECT y.year, g.geography, ROUND(f.value, 1) AS percent
FROM workspace.dev_gold.fact_health_indicator f
JOIN workspace.dev_gold.dim_geography   g ON f.geo_key = g.geo_key
JOIN workspace.dev_gold.dim_indicator   i ON f.indicator_key = i.indicator_key
JOIN workspace.dev_gold.dim_measure     m ON f.measure_key = m.measure_key
JOIN workspace.dev_gold.dim_demographic d ON f.demo_key = d.demo_key
JOIN workspace.dev_gold.dim_year        y ON f.year = y.year
WHERE i.indicator LIKE '%mental health%'
  AND m.characteristic LIKE 'Percent%'
  AND g.geography IN ('Ontario', 'Canada (excluding territories)')
  AND d.age_group LIKE 'Total%' AND d.sex LIKE 'Both%'
ORDER BY y.year, g.geography;